In [2]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 2.3 MB/s eta 0:00:01
   --------------------- ------------------ 1.0/2.0 MB 2.4 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.5 MB/s eta 0:00:01
   ------------------------------------ --- 1.8/2.0 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.3 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import networkx as nx
from faker import Faker
import random
from tqdm import tqdm

fake = Faker()
np.random.seed(42)
random.seed(42)

NUM_ACCOUNTS = 5000
NUM_MULES = 200

In [4]:
devices = []

device_ids = [f"D{i}" for i in range(4000)]

# Normal accounts
for acc in range(NUM_ACCOUNTS - NUM_MULES):
    devices.append({
        "device_id": random.choice(device_ids),
        "account_id": f"A{acc}",
        "device_type": random.choice(["Mobile", "Laptop"]),
        "device_os": random.choice(["Android", "iOS", "Windows"]),
        "device_model": fake.word(),
        "vpn_usage_flag": False,
        "device_trust_score": np.random.uniform(0.7, 1.0)
    })

# Mule accounts share devices
shared_devices = random.sample(device_ids, 50)

for acc in range(NUM_ACCOUNTS - NUM_MULES, NUM_ACCOUNTS):
    devices.append({
        "device_id": random.choice(shared_devices),
        "account_id": f"A{acc}",
        "device_type": "Mobile",
        "device_os": "Android",
        "device_model": fake.word(),
        "vpn_usage_flag": True,
        "device_trust_score": np.random.uniform(0.1, 0.5)
    })

devices_df = pd.DataFrame(devices)
devices_df.to_csv("devices.csv", index=False)

In [5]:
transactions = []
transaction_id = 0

for acc in range(NUM_ACCOUNTS - NUM_MULES):
    num_tx = np.random.randint(20, 60)

    for _ in range(num_tx):
        receiver = random.randint(0, NUM_ACCOUNTS - NUM_MULES)

        transactions.append({
            "transaction_id": f"T{transaction_id}",
            "sender_account_id": f"A{acc}",
            "receiver_account_id": f"A{receiver}",
            "amount": round(np.random.uniform(100, 10000), 2),
            "channel": random.choice(["Mobile", "UPI", "Web", "ATM"]),
            "timestamp": fake.date_time_this_year(),
            "is_fraud": 0,
            "fraud_type": "None"
        })
        transaction_id += 1

In [6]:
mule_accounts = [f"A{i}" for i in range(NUM_ACCOUNTS - NUM_MULES, NUM_ACCOUNTS)]
for mule in mule_accounts[:50]:
    victim = random.choice([f"A{i}" for i in range(2000)])

    for _ in range(5):
        transactions.append({
            "transaction_id": f"T{transaction_id}",
            "sender_account_id": victim,
            "receiver_account_id": mule,
            "amount": round(np.random.uniform(20000, 80000), 2),
            "channel": random.choice(["Mobile", "UPI"]),
            "timestamp": fake.date_time_this_year(),
            "is_fraud": 1,
            "fraud_type": "Star_Mule"
        })
        transaction_id += 1
ring = mule_accounts[50:100]

for i in range(len(ring)):
    transactions.append({
        "transaction_id": f"T{transaction_id}",
        "sender_account_id": ring[i],
        "receiver_account_id": ring[(i+1) % len(ring)],
        "amount": round(np.random.uniform(30000, 60000), 2),
        "channel": "UPI",
        "timestamp": fake.date_time_this_year(),
        "is_fraud": 1,
        "fraud_type": "Ring_Mule"
    })
    transaction_id += 1
chain = mule_accounts[100:150]

for i in range(len(chain) - 1):
    transactions.append({
        "transaction_id": f"T{transaction_id}",
        "sender_account_id": chain[i],
        "receiver_account_id": chain[i+1],
        "amount": round(np.random.uniform(50000, 90000), 2),
        "channel": "Mobile",
        "timestamp": fake.date_time_this_year(),
        "is_fraud": 1,
        "fraud_type": "Chain_Mule"
    })
    transaction_id += 1
for mule in mule_accounts[150:]:
    victim = random.choice([f"A{i}" for i in range(2000)])

    channels = ["Mobile", "UPI", "ATM"]

    for ch in channels:
        transactions.append({
            "transaction_id": f"T{transaction_id}",
            "sender_account_id": victim,
            "receiver_account_id": mule,
            "amount": round(np.random.uniform(40000, 70000), 2),
            "channel": ch,
            "timestamp": fake.date_time_this_year(),
            "is_fraud": 1,
            "fraud_type": "Cross_Channel"
        })
        transaction_id += 1
transactions_df = pd.DataFrame(transactions)
transactions_df.to_csv("transactions.csv", index=False)

In [7]:
logins = []

for acc in range(NUM_ACCOUNTS):
    num_logins = np.random.randint(10, 50)

    for _ in range(num_logins):
        logins.append({
            "account_id": f"A{acc}",
            "ip_address": fake.ipv4(),
            "country": fake.country(),
            "session_duration": np.random.randint(1, 60),
            "timestamp": fake.date_time_this_year()
        })

logins_df = pd.DataFrame(logins)
logins_df.to_csv("logins.csv", index=False)

In [8]:
G = nx.DiGraph()

for _, row in transactions_df.iterrows():
    G.add_edge(row["sender_account_id"], row["receiver_account_id"])
features = []

pagerank = nx.pagerank(G)
between = nx.betweenness_centrality(G)

for node in G.nodes():
    features.append({
        "account_id": node,
        "in_degree": G.in_degree(node),
        "out_degree": G.out_degree(node),
        "betweenness": between[node],
        "pagerank": pagerank[node]
    })

network_df = pd.DataFrame(features)
network_df.to_csv("network_features.csv", index=False)

CONVERTING TO NODES AND EDGES\

In [9]:
import pandas as pd

customers = pd.read_csv("customers.csv")
devices = pd.read_csv("devices.csv")
transactions = pd.read_csv("transactions.csv")
network = pd.read_csv("network_features.csv")

C:\Users\JHANANISHRI\AppData\Local\Temp\ipykernel_20548\519979824.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv("transactions.csv")


In [10]:
accounts = customers["account_id"].unique()
account_mapping = {acc: i for i, acc in enumerate(accounts)}
features = customers.merge(network, on="account_id", how="left")
features["customer_risk_category"] = features["customer_risk_category"].map({
    "low": 0,
    "medium": 1,
    "high": 2
})
feature_cols = [
    "credit_score",
    "average_monthly_balance",
    "customer_risk_category",
    "in_degree",
    "out_degree",
    "betweenness",
    "pagerank"
]

In [11]:
import torch

X = torch.tensor(features[feature_cols].fillna(0).values, dtype=torch.float)

In [12]:
edges = []

for _, row in transactions.iterrows():
    s = account_mapping.get(row["sender_account_id"])
    r = account_mapping.get(row["receiver_account_id"])

    if s is not None and r is not None:
        edges.append([s, r])

In [13]:
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [14]:
fraud_accounts = set(
    transactions[transactions["is_fraud"] == 1]["receiver_account_id"]
)
labels = []

for acc in accounts:
    labels.append(1 if acc in fraud_accounts else 0)

y = torch.tensor(labels, dtype=torch.long)

In [1]:
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html

  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)

   --- ------------------------------------  1/12 [pyparsing]
   --- ------------------------------------  1/12 [pyparsing]
   --- ------------------------------------  1/12 [pyparsing]
   --- ------------------------------------  1/12 [pyparsing]
   ---------- -----------------------------  3/12 [multidict]
   ---------------- -----------------------  5/12 [attrs]
   ---------------- -----------------------  5/12 [attrs]
   ---------------- -----------------------  5/12 [attrs]
   ----------------------- ----------------  7/12 [aiohappyeyeballs]
   -------------------------- -------------  8/12 [yarl]
   ------------------------------ ---------  9/12 [aiosignal]
   --------------------------------- ------ 10/12 [aiohttp]
   --------------------------------- ------ 10/12 [aiohttp]
   --------------------------------- ------ 10/12 [aiohttp]
   ------------

In [2]:
from torch_geometric.data import Data

graph_data = Data(x=X, edge_index=edge_index, y=y)

ModuleNotFoundError: No module named 'torch'